# Train a Custom "Hey Argus" Wake-Word Model

Trains an OpenWakeWord ONNX model on ~10,000 synthetic samples of
"Hey Argus" using Piper TTS across dozens of voices.

**Runtime:** ~30 min on a free Colab T4 GPU.
**Output:** `hey_argus.onnx` — drop into `~/.argus/wake_models/`

Click **Runtime → Run all** and grab a coffee.

## 1. Install dependencies

In [ ]:
!pip install -q openwakeword piper-tts torch torchaudio onnx onnxruntime

## 2. Generate synthetic "Hey Argus" samples

Uses Piper TTS to synthesise the phrase across many voices and speaking
rates. The variety teaches the model phonetic robustness.

In [ ]:
WAKE_PHRASE = "hey argus"
N_POSITIVE_SAMPLES = 10000
N_NEGATIVE_SAMPLES = 30000

from openwakeword.utils import download_models
download_models()

# Synthesize the wake-word samples across many Piper voices.
# This step takes ~12 min on a T4.
from openwakeword.data import generate_adversarial_texts, generate_clips
generate_clips(
    text=WAKE_PHRASE,
    n_samples=N_POSITIVE_SAMPLES,
    output_dir='positive_samples',
    voices='all',
    augment=True,            # add room reverb, noise, pitch shifts
)

# Negative samples — random English phrases that are NOT 'hey argus'.
# Includes phonetically similar distractors like 'hey arthur', 'hey hercules'
# so the model learns the discriminating boundary.
generate_adversarial_texts(
    target=WAKE_PHRASE,
    n=N_NEGATIVE_SAMPLES,
    output_dir='negative_samples',
)

## 3. (Optional) Add YOUR voice for a personalised boost

Skip this cell for a pure-synthetic model. If you have ~30 recordings of
yourself saying 'Hey Argus', upload them as a zip and they'll get 4x
weight in training.

In [ ]:
from google.colab import files
uploaded = files.upload()   # cancel if no real samples to upload
if uploaded:
    import zipfile, os
    for fname in uploaded:
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('real_positive_samples')
    # Duplicate them 4x so they outweigh the synthetic mass
    !for i in 1 2 3; do cp -r real_positive_samples positive_samples/real_${i}; done

## 4. Train the model

~100 epochs on a T4. Validation accuracy should hit ~98%.

In [ ]:
from openwakeword.train import train_custom_model

model_path = train_custom_model(
    target_phrase=WAKE_PHRASE,
    positive_samples_dir='positive_samples',
    negative_samples_dir='negative_samples',
    output_path='hey_argus.onnx',
    num_epochs=100,
    batch_size=128,
    learning_rate=1e-3,
)
print(f'Trained: {model_path}')

## 5. Test the model in-notebook

In [ ]:
from openwakeword.model import Model
import numpy as np, wave

model = Model(wakeword_models=['hey_argus.onnx'])

# Test on a held-out positive sample
import glob
test_wavs = glob.glob('positive_samples/*.wav')[:5]
for wav_path in test_wavs:
    with wave.open(wav_path, 'rb') as wf:
        audio = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16)
    # Feed in 80ms chunks
    chunk_size = 16000 * 80 // 1000
    max_score = 0.0
    for i in range(0, len(audio) - chunk_size, chunk_size):
        scores = model.predict(audio[i:i+chunk_size])
        max_score = max(max_score, max(scores.values()))
    print(f'  {wav_path}: max_score={max_score:.3f}', '✓' if max_score >= 0.5 else '✗')

## 6. Download the model

Drop the downloaded `hey_argus.onnx` into `~/.argus/wake_models/` and run:

```bash
argus listen --wake-model hey_argus
```

In [ ]:
from google.colab import files
files.download('hey_argus.onnx')